## SETUP AND INSTALLATION

In [50]:
"""# Install required libraries
# Run this cell only once!
import subprocess
import sys

libraries = [
    'openai',
    'python-dotenv',
    'requests'
]

for lib in libraries:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', lib, '-q'])
        print(f"✓ {lib} installed successfully")
    except Exception as e:
        print(f"✗ Error installing {lib}: {e}")

print("\n✅ All libraries installed!")"""

'# Install required libraries\n# Run this cell only once!\nimport subprocess\nimport sys\n\nlibraries = [\n    \'openai\',\n    \'python-dotenv\',\n    \'requests\'\n]\n\nfor lib in libraries:\n    try:\n        subprocess.check_call([sys.executable, \'-m\', \'pip\', \'install\', lib, \'-q\'])\n        print(f"✓ {lib} installed successfully")\n    except Exception as e:\n        print(f"✗ Error installing {lib}: {e}")\n\nprint("\n✅ All libraries installed!")'

## API KEY

In [51]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()  # for .env file

api_key_env = os.getenv('OPENROUTER_API_KEY')

if api_key_env:
    api_key = api_key_env
    print("✓ API key loaded from environment variable")
else:
    print("No API key found in environment.")
    print("Enter your OpenRouter API key below (it won't be displayed):")
    api_key = getpass("OpenRouter API Key: ")

    if api_key.startswith('sk-or-'):
        print("✓ API key format looks correct!")
    else:
        print("⚠️ Warning: API key doesn't start with 'sk-or-'. Check if it's correct.")

os.environ['OPENROUTER_API_KEY'] = api_key
print(f"\nAPI Key Set: {api_key[:8]}...{api_key[-3:]}")

✓ API key loaded from environment variable

API Key Set: sk-or-v1...3e4


## IMPORTING LIBRARIES

In [52]:
# Core libraries for API interaction
import requests
import json
import os
import time
from datetime import datetime

# For structured data handling
import pandas as pd

# For environment variable management
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

print("✓ All libraries imported successfully!")
print("\nLoaded libraries:")
print("  • requests - HTTP requests")
print("  • json - JSON data handling")
print("  • os - Operating system interface")
print("  • time - Time-related functions")
print("  • pandas - Data analysis")
print("  • dotenv - Environment variable management")

✓ All libraries imported successfully!

Loaded libraries:
  • requests - HTTP requests
  • json - JSON data handling
  • os - Operating system interface
  • time - Time-related functions
  • pandas - Data analysis
  • dotenv - Environment variable management


## OPENROUTER API: CALLS AND RESPONSE HANDLING

In [53]:
# API calls to OpenRouter

def call_openrouter_api(model, messages, temperature=0.7, max_tokens=500):
    """
    Call OpenRouter API with specified parameters.

    Args:
        model (str): Model ID (e.g., 'openai/gpt-4', 'anthropic/claude-3-opus')
        messages (list): List of message dictionaries with 'role' and 'content'
        temperature (float): Controls randomness (0=deterministic, 1=random)
        max_tokens (int): Maximum response length

    Returns:
        dict: API response or error information
    """
    api_key_env = os.getenv('OPENROUTER_API_KEY')
    if not api_key_env:
        return {"error": "API key not set. Please run the setup cell above."}

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key_env}",
        "HTTP-Referer": "https://github.com/adedamolaobadeji",
        "X-Title": "LLM Training Notebook",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    try:
        response = requests.post(url, json=payload, headers=headers, timeout=30)
        return response.json()
    except requests.exceptions.Timeout:
        return {"error": "Request timed out"}
    except requests.exceptions.RequestException as e:
        return {"error": f"Request failed: {str(e)}"}

print("✓ API helper function created!")
print("\nFunction: call_openrouter_api()")
print("  - Handles authentication")
print("  - Manages HTTP requests")
print("  - Returns JSON responses")

✓ API helper function created!

Function: call_openrouter_api()
  - Handles authentication
  - Manages HTTP requests
  - Returns JSON responses


In [54]:
# Response handling

def extract_response(api_response):
    """
    Extract the actual text response from API response.

    Args:
        api_response (dict): Response from OpenRouter API

    Returns:
        str: Extracted text or error message
    """

    if "error" in api_response:
        return f"❌ Error: {api_response['error']}"

    if "choices" in api_response and len(api_response["choices"]) > 0:
        return api_response["choices"][0]["message"]["content"]

    return "❌ Unexpected response format"

# Display API usage stats

def display_usage(api_response):
    """
    Display token usage statistics from API response.
    """
    if "usage" in api_response:
        usage = api_response["usage"]
        print("\n📊 API Usage Statistics:")
        print(f"  • Prompt tokens: {usage.get('prompt_tokens', 'N/A')}")
        print(f"  • Completion tokens: {usage.get('completion_tokens', 'N/A')}")
        print(f"  • Total tokens: {usage.get('total_tokens', 'N/A')}")

print("✓ Helper functions created!")

✓ Helper functions created!


## MODELS

In [55]:
# Test 1: Simple API call to check authentication
print("🧪 Test 1: Checking API Connection...\n")

# Create a simple test message
test_messages = [
    {"role": "user", "content": "Say 'Hello, I'm working!' and nothing else."}
]

# Call the API with a lightweight model
response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",  # Lightweight, fast, and cheap
    messages=test_messages,
    temperature=0.7,
    max_tokens=50
)

# Display results
if "error" not in response:
    print("✅ API Connection Successful!\n")
    print(f"Response: {extract_response(response)}")
    display_usage(response)
else:
    print(f"❌ Connection Failed: {response['error']}")
    print("\nTroubleshooting:")
    print("1. Check if your API key is correct")
    print("2. Verify you have API credits (free tier has limits)")
    print("3. Check your internet connection")

🧪 Test 1: Checking API Connection...

✅ API Connection Successful!

Response: Hello, I'm working!

📊 API Usage Statistics:
  • Prompt tokens: 19
  • Completion tokens: 6
  • Total tokens: 25


In [56]:
# Test 2: Compare different models
print("\n🧪 Test 2: Testing Different Models...\n")
print("Sending the same prompt to different models...\n")

test_prompt = "Explain machine learning in 2 sentences."

models_to_test = [
    ("openai/gpt-3.5-turbo", "GPT-3.5 Turbo"),
    ("mistralai/mistral-small", "Mistral Small"),
]

results = []

for model_id, model_name in models_to_test:
    print(f"Testing {model_name}...")

    response = call_openrouter_api(
        model=model_id,
        messages=[{"role": "user", "content": test_prompt}],
        temperature=0.7,
        max_tokens=100
    )

    if "error" not in response:
        answer = extract_response(response)
        tokens = response.get("usage", {}).get("total_tokens", "N/A")
        results.append({
            "Model": model_name,
            "Response": answer[:100] + "..." if len(answer) > 100 else answer,
            "Tokens Used": tokens
        })
        print(f"  ✓ Success\n")
    else:
        print(f"  ✗ Error: {response['error']}\n")
        results.append({
            "Model": model_name,
            "Response": "Error",
            "Tokens Used": "N/A"
        })

# Display results in a table
if results:
    df = pd.DataFrame(results)
    print("\n📊 Model Comparison:")
    print(df.to_string(index=False))


🧪 Test 2: Testing Different Models...

Sending the same prompt to different models...

Testing GPT-3.5 Turbo...
  ✓ Success

Testing Mistral Small...
  ✗ Error: {'message': 'No endpoints found for mistralai/mistral-small.', 'code': 404}


📊 Model Comparison:
        Model                                                                                                Response Tokens Used
GPT-3.5 Turbo Machine learning is a subset of artificial intelligence that allows systems to learn and improve fro...          55
Mistral Small                                                                                                   Error         N/A


In [57]:
# Test 2a Get model list from OpenRouter, due to error with Mistral.
print("\n🧪 Test 2a: Fetching Available Models...\n")

response = requests.get(
    "https://openrouter.ai/api/v1/models",
    headers={"Authorization": f"Bearer {os.getenv('OPENROUTER_API_KEY')}"}
)

data = response.json()

# Extract only model IDs
model_ids = [model["id"] for model in data.get("data", [])]

print(f"✅ Found {len(model_ids)} models:\n")

for model_id in model_ids[:30]:  # limit to first n models.
    print(model_id)


🧪 Test 2a: Fetching Available Models...

✅ Found 345 models:

anthropic/claude-opus-4.7
openrouter/elephant-alpha
anthropic/claude-opus-4.6-fast
z-ai/glm-5.1
google/gemma-4-26b-a4b-it:free
google/gemma-4-26b-a4b-it
google/gemma-4-31b-it:free
google/gemma-4-31b-it
qwen/qwen3.6-plus
z-ai/glm-5v-turbo
arcee-ai/trinity-large-thinking
x-ai/grok-4.20-multi-agent
x-ai/grok-4.20
google/lyria-3-pro-preview
google/lyria-3-clip-preview
kwaipilot/kat-coder-pro-v2
rekaai/reka-edge
xiaomi/mimo-v2-omni
xiaomi/mimo-v2-pro
minimax/minimax-m2.7
openai/gpt-5.4-nano
openai/gpt-5.4-mini
mistralai/mistral-small-2603
z-ai/glm-5-turbo
nvidia/nemotron-3-super-120b-a12b:free
nvidia/nemotron-3-super-120b-a12b
bytedance-seed/seed-2.0-lite
qwen/qwen3.5-9b
openai/gpt-5.4-pro
openai/gpt-5.4


In [58]:
# Test 2b: Compare different models. Use model list to avoid ID mismatch or use of deprecated models.
print("\n🧪 Test 2b: Testing Different Models...\n")
print("Sending the same prompt to different models...\n")

test_prompt = "Explain machine learning in 1 sentence."

models_to_test = [
    ("openai/gpt-3.5-turbo", "GPT-3.5 Turbo"),
    ("x-ai/grok-4.20", "grok-4.20"),
    ("google/gemma-4-31b-it", "Gemma 4 31B IT"),
    ("xiaomi/mimo-v2-omni", "MIMO v2 Omni"),
    ("qwen/qwen3.6-plus", "Qwen 3.6 Plus"),
    ("deepseek/deepseek-r1", "DeepSeek R1") 

]

results = []

for model_id, model_name in models_to_test:
    print(f"Testing {model_name}...")

    response = call_openrouter_api(
        model=model_id,
        messages=[{"role": "user", "content": test_prompt}],
        temperature=0.7,
        max_tokens=80
    )

    if "error" not in response:
        answer = extract_response(response)
        tokens = response.get("usage", {}).get("total_tokens", "N/A")
        results.append({
            "Model": model_name,
            "Response": answer,
            "Tokens Used": tokens
        })
        print(f"  ✓ Success\n")
    else:
        print(f"  ✗ Error: {response['error']}\n")
        results.append({
            "Model": model_name,
            "Response": "Error",
            "Tokens Used": "N/A"
        })

# Display results in a table
if results:
    pd.set_option('display.max_colwidth', None)
    df = pd.DataFrame(results)
    print("\n📊 Model Comparison:")
    display(df)


🧪 Test 2b: Testing Different Models...

Sending the same prompt to different models...

Testing GPT-3.5 Turbo...
  ✓ Success

Testing grok-4.20...
  ✓ Success

Testing Gemma 4 31B IT...
  ✓ Success

Testing MIMO v2 Omni...
  ✓ Success

Testing Qwen 3.6 Plus...
  ✓ Success

Testing DeepSeek R1...
  ✓ Success


📊 Model Comparison:


,Model,Response,Tokens Used
0,GPT-3.5 Turbo,Machine learning is a branch of artificial intelligence that uses algorithms to enable machines to learn from data and make decisions or predictions without being explicitly programmed.,44
1,grok-4.20,Machine learning is a branch of AI where computers learn patterns and make predictions from data without being explicitly programmed for each task.,154
2,Gemma 4 31B IT,Machine learning is a field of artificial intelligence where computers use data to identify patterns and improve their performance on a specific task without being explicitly programmed for every outcome.,53
3,MIMO v2 Omni,NaN,339
4,Qwen 3.6 Plus,"Machine learning is a branch of artificial intelligence that enables computers to automatically learn from data, identify patterns, and improve at tasks without being explicitly programmed.",519
5,DeepSeek R1,Machine learning is the development of algorithms that enable computers to learn patterns from data and make predictions or decisions without being explicitly programmed for each task.,149


In [59]:
# MIMO v2 Omni NaN error check
response = call_openrouter_api(
    model="xiaomi/mimo-v2-omni",
    messages=[{"role": "user", "content": test_prompt}],
    temperature=0.7,
    max_tokens=80
)

print(json.dumps(response, indent=2))

{
  "id": "gen-1776461571-BKfbam7jxbA7z8Qt92E3",
  "object": "chat.completion",
  "created": 1776461571,
  "model": "xiaomi/mimo-v2-omni-20260318",
  "provider": "Xiaomi",
  "system_fingerprint": null,
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "length",
      "native_finish_reason": "length",
      "message": {
        "role": "assistant",
        "content": null,
        "refusal": null,
        "reasoning": "Hmm, the user wants a one-sentence explanation of machine learning. This is a straightforward request for a concise definition. \n\nI should focus on capturing the core idea: learning from data to improve performance on tasks. The explanation needs to be simple yet accurate, avoiding technical jargon since the user didn't ask for depth. \n\nThe key elements to include are: learning from data, pattern recognition",
        "reasoning_details": [
          {
            "type": "reasoning.text",
            "text": "Hmm, the user wants a 

Why did Mimo display NaN under response?: 
It spent all max 80 tokens reasoning instead of writing a response. extract_response() returned NaN hence there was nothing in content to extract. Under 'reasoning',  it started forming an answer but got cut off mid-sentence. It ran out of tokens before it could finish thinking and write the actual response?

Mimo is a reasoning model. How is the token allocation calculated? Does it exhaust all the tokens for reasoning before response completion?.
Recommended fix is to increase max tokens? hence increased cost.

In [60]:
# Check another reasoning model: DeepSeek R1 
    
response = call_openrouter_api(
    model="deepseek/deepseek-r1",
    messages=[{"role": "user", "content": test_prompt}],
    temperature=0.7,
    max_tokens=80
)

print(json.dumps(response, indent=2))

{
  "id": "gen-1776461572-cxXszDn83E6efcyeTFC2",
  "object": "chat.completion",
  "created": 1776461572,
  "model": "deepseek/deepseek-r1",
  "provider": "Novita",
  "system_fingerprint": null,
  "choices": [
    {
      "index": 0,
      "logprobs": null,
      "finish_reason": "stop",
      "native_finish_reason": "stop",
      "message": {
        "role": "assistant",
        "content": "Machine learning is a field of artificial intelligence where systems use algorithms to learn from data patterns and make decisions without being explicitly programmed for every scenario.",
        "refusal": null,
        "reasoning": "Hmm, the user wants a one-sentence explanation of machine learning. This needs to be concise yet comprehensive. \n\nMachine learning is a broad field, so I should capture its essence: the ability of systems to learn from data without explicit programming. \n\nThe key elements are algorithms, data, pattern recognition, and decision-making. I can structure it as \"a fie

DeepSeek printed a response at max_tokens=80 but Mimo didn't. Both are reasoning models and reasoning tokens was 98 and 79 respectively. 

How was deepseek able to generate a response even after exceeding the max tokens in the reasoning?
Is reasoning_token counted in max_token?
How do you know how token allocation is programmed for each model? Via documentation?

In [61]:
# Function to inspect tokens

def inspect_model_tokens(model_id, prompt, max_tokens=80, temperature=0.7):
    print(f"\n🔍 Inspecting model: {model_id}")
    print("-" * 60)

    response = call_openrouter_api(
        model=model_id,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )

    # Handle errors
    if "error" in response:
        print(f"❌ Error: {response['error']}")
        return

    # Extract response text
    answer = extract_response(response)

    # Extract usage safely
    usage = response.get("usage", {})
    prompt_tokens = usage.get("prompt_tokens", "N/A")
    completion_tokens = usage.get("completion_tokens", "N/A")
    total_tokens = usage.get("total_tokens", "N/A")
    reasoning_tokens = usage.get("completion_tokens_details", {}).get("reasoning_tokens", "N/A")

    # Print results
    print("🧠 Response:")
    print(answer)

    print("\n📊 Token Usage:")
    print(f"  Prompt tokens: {prompt_tokens}")
    print(f"  Completion tokens: {completion_tokens}")
    print(f"  Total tokens: {total_tokens}")
    print(f"  Reasoning tokens: {reasoning_tokens}")

test_prompt = "Tell me about TravelRadar."

models = [
    "openai/gpt-3.5-turbo",
    "deepseek/deepseek-r1",
    "xiaomi/mimo-v2-omni",
    "x-ai/grok-4.20"
]

for model in models:
    inspect_model_tokens(model, test_prompt, max_tokens=80)


🔍 Inspecting model: openai/gpt-3.5-turbo
------------------------------------------------------------
🧠 Response:
TravelRadar is a travel website and app that helps users find the best deals on flights, hotels, and car rentals. It aggregates information from various travel booking websites and allows users to compare prices and options in one place. TravelRadar also offers personalized recommendations based on user preferences and budget constraints. The platform aims to make travel planning easier and more affordable for users by providing access to a wide range

📊 Token Usage:
  Prompt tokens: 14
  Completion tokens: 80
  Total tokens: 94
  Reasoning tokens: 0

🔍 Inspecting model: deepseek/deepseek-r1
------------------------------------------------------------
🧠 Response:
TravelRadar is a **flight tracking and aviation data platform** that provides real-time information about flights worldwide. It's commonly used by aviation enthusiasts, travelers, and professionals to monitor flig

## PROMPT ENGINEERING

In [62]:
print("📝 Prompt Engineering Examples\n")
print("="*60)

print("\n❌ WEAK PROMPT:")
weak_prompt = "Tell me about Python"
print(f"Prompt: '{weak_prompt}'")

response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": weak_prompt}],
    temperature=0.7,
    max_tokens=200
)
print(f"\nResponse: {extract_response(response)}")

print("\n" + "="*60)
print("\n✅ STRONG PROMPT:")
strong_prompt = """You are an expert Python instructor.
Explain Python for beginners in 3-4 sentences.
Focus on: what it is, why it's popular, and one key benefit.
Use simple language."""
print(f"Prompt: '{strong_prompt}'")

response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": strong_prompt}],
    temperature=0.7,
    max_tokens=200
)
print(f"\nResponse: {extract_response(response)}")

print("\n" + "="*60)
print("\n💬 Notice the difference?")
print("The strong prompt gets more focused and relevant responses!")

📝 Prompt Engineering Examples


❌ WEAK PROMPT:
Prompt: 'Tell me about Python'

Response: Python is a high-level, interpreted programming language that was created by Guido van Rossum in the late 1980s. It is known for its simplicity and readability, making it a popular choice for beginners and experienced programmers alike.

Python is widely used in various fields such as web development, data analysis, artificial intelligence, scientific computing, and more. It has a large standard library that provides a wide range of modules and packages for different tasks.

Some key features of Python include dynamic typing, automatic memory management, and support for multiple programming paradigms such as procedural, object-oriented, and functional programming.

Python is also known for its extensive community support, with a large number of libraries and frameworks available for developers to use. Some popular libraries include NumPy, Pandas, Django, Flask, and TensorFlow.

Overall, Python is a

In [63]:
# Example 2: Few-Shot Learning (Providing Examples)

print("\n\n🎓 Few-Shot Learning: Teaching by Example\n")
print("="*60)

few_shot_prompt = """Convert user requests into structured JSON.

Examples:
User: "I want a red car"
JSON: {"item": "car", "color": "red"}

User: "Get me a large coffee"
JSON: {"item": "coffee", "size": "large"}

Now convert this:
User: "I need a small blue notebook"""

response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0.3,  # Lower temperature for more consistent outputs
    max_tokens=100
)

print(f"Prompt:\n{few_shot_prompt}")
print(f"\n\nResponse:\n{extract_response(response)}")
print("\n" + "="*60)
print("\n💡 By providing examples, the model learns the exact format you want!")



🎓 Few-Shot Learning: Teaching by Example

Prompt:
Convert user requests into structured JSON.

Examples:
User: "I want a red car"
JSON: {"item": "car", "color": "red"}

User: "Get me a large coffee"
JSON: {"item": "coffee", "size": "large"}

Now convert this:
User: "I need a small blue notebook


Response:
JSON: {"item": "notebook", "size": "small", "color": "blue"}


💡 By providing examples, the model learns the exact format you want!


In [64]:
# Example 2: Few-Shot Learning (Providing Examples)

print("\n\n🎓 Few-Shot Learning: Teaching by Example\n")
print("="*60)

# --- Add more few-shot prompts here ---
few_shot_prompts = {
    "JSON Converter": """Convert user requests into structured JSON.

Examples:
User: "I want a red car"
JSON: {"item": "car", "color": "red"}

User: "Get me a large coffee"
JSON: {"item": "coffee", "size": "large"}

Now convert this:
User: "I need a small blue notebook"
JSON:""",

    "Sentiment Labeller": """Label the sentiment of each sentence as Positive, Negative, or Neutral.

Examples:
Sentence: "I love this product!"
Label: Positive

Sentence: "This is the worst experience ever."
Label: Negative

Now label this:
Sentence: "The flight was okay, nothing special."
Label:""",

    "Travel Intent Extractor": """Extract travel intent from user messages into structured JSON.

Examples:
User: "I want to fly to Paris next Friday"
JSON: {"destination": "Paris", "date": "next Friday", "mode": "flight"}

User: "Book me a train to Manchester tomorrow"
JSON: {"destination": "Manchester", "date": "tomorrow", "mode": "train"}

Now convert this:
User: "I need a flight to Dubai in two weeks"
JSON:"""
}

# --- Add more models here ---
models = [
    ("openai/gpt-3.5-turbo", "GPT-3.5 Turbo"),
    ("x-ai/grok-4.20", "grok-4.20"),
    ("deepseek/deepseek-r1", "DeepSeek R1"),
]

# --- Run all combinations ---
for prompt_name, prompt_text in few_shot_prompts.items():
    print(f"\n📋 Prompt Type: {prompt_name}")
    print("=" * 60)
    print(f"📝 Full Prompt:\n{prompt_text.strip()}")
    print("-" * 60)

    for model_id, model_name in models:
        print(f"\n🤖 {model_name}")
        print("-" * 40)

        response = call_openrouter_api(
            model=model_id,
            messages=[{"role": "user", "content": prompt_text}],
            temperature=0.3,
            max_tokens=200
        )

        print(f"Response: {extract_response(response)}")

    print("\n" + "=" * 60)



🎓 Few-Shot Learning: Teaching by Example


📋 Prompt Type: JSON Converter
📝 Full Prompt:
Convert user requests into structured JSON.

Examples:
User: "I want a red car"
JSON: {"item": "car", "color": "red"}

User: "Get me a large coffee"
JSON: {"item": "coffee", "size": "large"}

Now convert this:
User: "I need a small blue notebook"
JSON:
------------------------------------------------------------

🤖 GPT-3.5 Turbo
----------------------------------------
Response: {"item": "notebook", "size": "small", "color": "blue"}

🤖 grok-4.20
----------------------------------------
Response: ```json
{"item": "notebook", "size": "small", "color": "blue"}
```

🤖 DeepSeek R1
----------------------------------------
Response: {"item": "notebook", "size": "small", "color": "blue"}


📋 Prompt Type: Sentiment Labeller
📝 Full Prompt:
Label the sentiment of each sentence as Positive, Negative, or Neutral.

Examples:
Sentence: "I love this product!"
Label: Positive

Sentence: "This is the worst experien

In [65]:
# Example 3: Role-Playing (Setting Context)

print("\n\n🎭 Role-Playing: Setting the Model's Persona\n")
print("="*60)

role_prompt = """You are a sarcastic tech support representative.
A customer asks: "Why is my computer slow?"
Respond in a sarcastic but helpful way (max 3 sentences)."""

response = call_openrouter_api(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": role_prompt}],
    temperature=0.8,  # Higher for more creative/varied responses
    max_tokens=150
)

print(f"Prompt: {role_prompt}")
print(f"\nResponse:\n{extract_response(response)}")

print("\n" + "="*60)
print("\n💡 Assigning a role makes the model respond in that style!")



🎭 Role-Playing: Setting the Model's Persona

Prompt: You are a sarcastic tech support representative.
A customer asks: "Why is my computer slow?"
Respond in a sarcastic but helpful way (max 3 sentences).

Response:
Oh, I'm sure your computer is just taking its time to enjoy the scenery. Have you tried feeding it some more RAM or cleaning out its virtual clutter? That might help speed things up a bit.


💡 Assigning a role makes the model respond in that style!


## EXPERIMENTING PARAMETERS

In [66]:
# Experiment 1: Temperature Effects

print("🔬 Temperature Experiment: How does it affect creativity?\n")
print("="*70)

temperature_prompt = "Complete this sentence: 'The future of AI is...'"

temperatures = [0.1, 0.7, 1.5]

for temp in temperatures:
    print(f"\n📊 Temperature = {temp}")
    print("-" * 70)

    # Get 2 responses to show variability
    for i in range(2):
        response = call_openrouter_api(
            model="openai/gpt-3.5-turbo",
            messages=[{"role": "user", "content": temperature_prompt}],
            temperature=temp,
            max_tokens=50
        )
        print(f"Response {i+1}: {extract_response(response)}")

print("\n" + "="*70)
print("\n💡 Observations:")
print("• Low temperature (0.1): More consistent, similar responses")
print("• Medium temperature (0.7): Balanced, varied but logical")
print("• High temperature (1.5): Very creative, unpredictable")

🔬 Temperature Experiment: How does it affect creativity?


📊 Temperature = 0.1
----------------------------------------------------------------------
Response 1: uncertain, but full of potential for both positive and negative impacts on society.
Response 2: uncertain, but full of potential for both positive and negative impacts on society.

📊 Temperature = 0.7
----------------------------------------------------------------------
Response 1: uncertain, as advancements in technology continue to evolve and ethical considerations are still being debated.
Response 2: unpredictable and full of potential.

📊 Temperature = 1.5
----------------------------------------------------------------------
Response 1: intriguing and filled with endless possibilities as technology continues to advance and innovate.
Response 2: unpredictable and boundless in its potential revolutionary impacts on society.


💡 Observations:
• Low temperature (0.1): More consistent, similar responses
• Medium temperature (

In [67]:
# Experiment 2: Max Tokens Effect

print("\n\n🔬 Max Tokens Experiment: Controlling Response Length\n")
print("="*70)

length_prompt = "What is blockchain technology? Explain in detail."

token_limits = [50, 150, 300]

for max_tok in token_limits:
    print(f"\n📝 Max Tokens = {max_tok}")
    print("-" * 70)

    response = call_openrouter_api(
        model="openai/gpt-3.5-turbo",
        messages=[{"role": "user", "content": length_prompt}],
        temperature=0.7,
        max_tokens=max_tok
    )

    answer = extract_response(response)
    actual_tokens = response.get("usage", {}).get("completion_tokens", "N/A")

    print(f"Response (truncated):")
    print(answer[:150] + "..." if len(answer) > 150 else answer)
    print(f"\nActual tokens used: {actual_tokens}")

print("\n" + "="*70)
print("\n💡 Key Insight:")
print("max_tokens caps the response length. Longer responses = higher cost!")



🔬 Max Tokens Experiment: Controlling Response Length


📝 Max Tokens = 50
----------------------------------------------------------------------
Response (truncated):
Blockchain technology is a decentralized, distributed ledger system that enables secure, transparent, and tamper-proof recording of transactions acros...

Actual tokens used: 50

📝 Max Tokens = 150
----------------------------------------------------------------------
Response (truncated):
Blockchain technology is a decentralized, distributed ledger system that records transactions across a network of computers. Each transaction is group...

Actual tokens used: 150

📝 Max Tokens = 300
----------------------------------------------------------------------
Response (truncated):
Blockchain technology is a decentralized, distributed ledger system that allows for secure and transparent record-keeping of transactions across a net...

Actual tokens used: 300


💡 Key Insight:
max_tokens caps the response length. Longer responses

In [68]:
# Experiment 3: Parameter Comparison

print("\n\n🔬 Side-by-Side Parameter Comparison\n")
print("="*70)

comparison_prompt = "Write a creative product name for a coffee shop AI app"

configs = [
    {"temp": 0.3, "max_tok": 30, "label": "Conservative"},
    {"temp": 0.7, "max_tok": 50, "label": "Balanced"},
    {"temp": 1.2, "max_tok": 75, "label": "Creative"},
]

for config in configs:
    print(f"\n🎯 {config['label']} (temp={config['temp']}, max_tokens={config['max_tok']})")
    print("-" * 70)

    response = call_openrouter_api(
        model="openai/gpt-3.5-turbo",
        messages=[{"role": "user", "content": comparison_prompt}],
        temperature=config['temp'],
        max_tokens=config['max_tok']
    )

    print(f"Response: {extract_response(response)}")
    display_usage(response)

print("\n" + "="*70)



🔬 Side-by-Side Parameter Comparison


🎯 Conservative (temp=0.3, max_tokens=30)
----------------------------------------------------------------------
Response: BrewGenius

📊 API Usage Statistics:
  • Prompt tokens: 18
  • Completion tokens: 4
  • Total tokens: 22

🎯 Balanced (temp=0.7, max_tokens=50)
----------------------------------------------------------------------
Response: "CaféGenie: Your Personal Barista"

📊 API Usage Statistics:
  • Prompt tokens: 18
  • Completion tokens: 11
  • Total tokens: 29

🎯 Creative (temp=1.2, max_tokens=75)
----------------------------------------------------------------------
Response: BrewSavvy

📊 API Usage Statistics:
  • Prompt tokens: 18
  • Completion tokens: 5
  • Total tokens: 23



## ERROR HANDLING AND RATE LIMITS

In [69]:
# Robust API call with error handling and retry logic

def call_openrouter_with_retry(model, messages, temperature=0.7, max_tokens=500, max_retries=3):
    """
    Call OpenRouter API with retry logic and comprehensive error handling.

    Args:
        model (str): Model ID
        messages (list): Message list
        temperature (float): Temperature parameter
        max_tokens (int): Max tokens in response
        max_retries (int): Number of retry attempts

    Returns:
        dict: API response or error details
    """

    api_key = os.environ.get('OPENROUTER_API_KEY')
    if not api_key:
        return {
            "error": "API key not found",
            "details": "Set OPENROUTER_API_KEY environment variable"
        }

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "HTTP-Referer": "https://github.com/adedamolaobadeji",
        "X-Title": "LLM Training Notebook",
        "Content-Type": "application/json"
    }

    payload = {
        "model": model,
        "messages": messages,
        "temperature": temperature,
        "max_tokens": max_tokens
    }

    for attempt in range(max_retries):
        try:
            response = requests.post(url, json=payload, headers=headers, timeout=30)

            # Check for rate limiting
            if response.status_code == 429:
                wait_time = int(response.headers.get('Retry-After', 5))
                print(f"⚠️  Rate limited. Waiting {wait_time} seconds... (Attempt {attempt+1}/{max_retries})")
                time.sleep(wait_time)
                continue

            # Check for authentication errors
            if response.status_code == 401:
                return {
                    "error": "Authentication failed",
                    "details": "Invalid API key. Check your credentials."
                }

            # Check for server errors
            if response.status_code >= 500:
                if attempt < max_retries - 1:
                    wait_time = 2 ** attempt  # Exponential backoff
                    print(f"⚠️  Server error. Retrying in {wait_time} seconds... (Attempt {attempt+1}/{max_retries})")
                    time.sleep(wait_time)
                    continue

            # Success
            response.raise_for_status()
            return response.json()

        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                print(f"⚠️  Request timeout. Retrying... (Attempt {attempt+1}/{max_retries})")
                time.sleep(1)
            else:
                return {"error": "Request timeout after retries"}

        except requests.exceptions.RequestException as e:
            return {"error": f"Request failed: {str(e)}"}

    return {"error": "Max retries exceeded"}

print("✓ Robust API function with error handling created!")

✓ Robust API function with error handling created!


In [70]:
# Test the error handling

print("\n🧪 Testing Error Handling\n")
print("="*70)

print("\n✅ Test 1: Valid Request")
response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Say 'Success!' briefly"}],
    max_tokens=50
)

if "error" not in response:
    print(f"Response: {extract_response(response)}")
else:
    print(f"Error: {response['error']}")

print("\n" + "="*70)

print("\n❌ Test 2: Invalid API Key (Simulated)")
bad_key = os.environ['OPENROUTER_API_KEY']
os.environ['OPENROUTER_API_KEY'] = "invalid-key"

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": "Test"}],
    max_tokens=50
)

if "error" in response:
    print(f"Error caught: {response['error']}")
    print(f"Details: {response.get('details', 'N/A')}")

# Restore the key
os.environ['OPENROUTER_API_KEY'] = bad_key

print("\n" + "="*70)


🧪 Testing Error Handling


✅ Test 1: Valid Request


Response: Success!


❌ Test 2: Invalid API Key (Simulated)
Error caught: Authentication failed
Details: Invalid API key. Check your credentials.



## ADVANCED USE CASES

In [71]:
# Use Case 1: Text Summarization

print("\n📚 Use Case 1: Text Summarization\n")
print("="*70)

long_text = """The global tourism industry, which accounts for roughly 10% of world GDP and supports hundreds of millions of jobs, was brought to a near standstill by the COVID-19 pandemic and is still recovering in many regions. The pandemic highlighted the vulnerability of the tourism sector to global crises, but it also accelerated the adoption of digital technologies and sustainable practices. As travel resumes, there is a growing emphasis on responsible tourism that minimizes environmental impact and supports local communities. The future of tourism is likely to be shaped by a combination of health safety measures, technological innovations, and a shift towards more meaningful and sustainable travel experiences."""

summarization_prompt = f"""Summarize this text in 1-2 sentences:

{long_text}

Summary:"""

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": summarization_prompt}],
    temperature=0.3,
    max_tokens=100
)

print("Original text:")
print(f"{long_text}\n")
print("-" * 70)
print("\nSummary:")
print(extract_response(response))
print("\n" + "="*70)


📚 Use Case 1: Text Summarization

Original text:
The global tourism industry, which accounts for roughly 10% of world GDP and supports hundreds of millions of jobs, was brought to a near standstill by the COVID-19 pandemic and is still recovering in many regions. The pandemic highlighted the vulnerability of the tourism sector to global crises, but it also accelerated the adoption of digital technologies and sustainable practices. As travel resumes, there is a growing emphasis on responsible tourism that minimizes environmental impact and supports local communities. The future of tourism is likely to be shaped by a combination of health safety measures, technological innovations, and a shift towards more meaningful and sustainable travel experiences.

----------------------------------------------------------------------

Summary:
The COVID-19 pandemic severely impacted the global tourism industry, leading to a focus on digital technologies and sustainable practices. As travel resumes

In [76]:
# Use Case 2: Translation

print("\n🌍 Use Case 2: Language Translation\n")
print("="*70)

english_text = "Hello! How are you today? I hope you're having a great day."

translation_prompt = f"""Translate this English text to mandarin chinese, igbo, yoruba and french:

'{english_text}'

Spanish translation:"""

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": translation_prompt}],
    temperature=0.3,
    max_tokens=200
)

print(f"English: {english_text}\n")
print("-" * 70)
print(f"\nSpanish: {extract_response(response)}")
print("\n" + "="*70)


🌍 Use Case 2: Language Translation

English: Hello! How are you today? I hope you're having a great day.

----------------------------------------------------------------------

Spanish: ¡Hola! ¿Cómo estás hoy? Espero que estés teniendo un gran día. 

Mandarin Chinese translation: 你好！你今天好吗？希望你有一个美好的一天。

Igbo translation: Kedụ ka ịmere gị? A dịghị mma nke ụbọchị.

Yoruba translation: Bawo ni o? Nibo ni o wa ni ojo kan?

French translation: Bonjour! Comment ça va aujourd'hui? J'espère que tu passes une bonne journée.



In [73]:
# Use Case 3: Code Generation

print("\n💻 Use Case 3: Code Generation\n")
print("="*70)

code_prompt = """Write a Python function that:
1. Takes a list of numbers as input
2. Returns the sum of all even numbers
3. Includes error handling

Provide only the code, no explanation."""

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": code_prompt}],
    temperature=0.3,
    max_tokens=250
)

print(f"Request: {code_prompt}\n")
print("-" * 70)
print("\nGenerated Code:")
print(extract_response(response))
print("\n" + "="*70)


💻 Use Case 3: Code Generation

Request: Write a Python function that:
1. Takes a list of numbers as input
2. Returns the sum of all even numbers
3. Includes error handling

Provide only the code, no explanation.

----------------------------------------------------------------------

Generated Code:
def sum_even_numbers(numbers):
    try:
        return sum(num for num in numbers if num % 2 == 0)
    except TypeError:
        return "Error: Input must be a list of numbers"



In [74]:
# Use Case 4: Multi-turn Conversation

print("\n💬 Use Case 4: Multi-Turn Conversation\n")
print("="*70)

print("Let's have a conversation with the AI!\n")

conversation_history = [
    {"role": "user", "content": "What's your favorite programming language?"}
]

print("User: What's your favorite programming language?")

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=conversation_history,
    temperature=0.7,
    max_tokens=150
)

assistant_response = extract_response(response)
print(f"\nAssistant: {assistant_response}")

# Add assistant response to history
conversation_history.append({"role": "assistant", "content": assistant_response})

# Continue conversation
conversation_history.append({"role": "user", "content": "Why do you like that language?"})

print("\n" + "-" * 70)
print("\nUser: Why do you like that language?")

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=conversation_history,
    temperature=0.7,
    max_tokens=150
)

assistant_response = extract_response(response)
print(f"\nAssistant: {assistant_response}")

print("\n" + "="*70)
print("\n💡 Notice: The AI remembers the conversation context!")
print("This is how chatbots maintain coherent conversations.")


💬 Use Case 4: Multi-Turn Conversation

Let's have a conversation with the AI!

User: What's your favorite programming language?

Assistant: As an AI, I do not have personal preferences. However, some popular programming languages among developers are Python, JavaScript, Java, C++, and Ruby.

----------------------------------------------------------------------

User: Why do you like that language?

Assistant: As an AI, I do not have personal preferences or the capacity to "like" any programming language. However, I am proficient in understanding and generating code in various languages to assist users with their queries and tasks.


💡 Notice: The AI remembers the conversation context!
This is how chatbots maintain coherent conversations.


In [78]:
# Use Case 5: Content Analysis & Sentiment

print("\n📊 Use Case 5: Sentiment Analysis & Content Insights\n")
print("="*70)

reviews = [
    "This product is amazing! Best purchase I've made.",
    "Terrible quality. Broke after one week. Very disappointed.",
    "It's okay. Does what it's supposed to do."
]

analysis_prompt = """Analyze the sentiment of these reviews. For each, provide:
1. Sentiment (Positive/Negative/Neutral)
2. Score (1-10)
3. Key emotion

Format as: Review | Sentiment | Score | Key Emotion

Reviews:
{}

Analysis:""".format("\n".join(reviews))

response = call_openrouter_with_retry(
    model="openai/gpt-3.5-turbo",
    messages=[{"role": "user", "content": analysis_prompt}],
    temperature=0.3,
    max_tokens=350
)

print("Reviews to analyze:")
for i, review in enumerate(reviews, 1):
    print(f"{i}. {review}")

print("\n" + "-" * 70)
print("\nAnalysis Results:")
print(extract_response(response))
print("\n" + "="*70)


📊 Use Case 5: Sentiment Analysis & Content Insights

Reviews to analyze:
1. This product is amazing! Best purchase I've made.
2. Terrible quality. Broke after one week. Very disappointed.
3. It's okay. Does what it's supposed to do.

----------------------------------------------------------------------

Analysis Results:
1. Review: This product is amazing! Best purchase I've made. | Sentiment: Positive | Score: 9 | Key Emotion: Excitement
2. Review: Terrible quality. Broke after one week. Very disappointed. | Sentiment: Negative | Score: 2 | Key Emotion: Disappointment
3. Review: It's okay. Does what it's supposed to do. | Sentiment: Neutral | Score: 5 | Key Emotion: Indifference

